# Feature Importance Analysis - PAS Scoring Model

Interactive Jupyter notebook for analyzing variable importance, sensitivity, and optimization opportunities.

## How to Use
1. Run each cell in order (Shift+Enter)
2. Modify weights in the **Configuration** section to test scenarios
3. Re-run the analysis cells to see impact
4. Results automatically save to your output folder

**Results will be saved to**: `C:\Box\Box\BOX Subhashree Singh\Business\PAS\Output`

## 📁 OUTPUT CONFIGURATION

**MODIFY THIS CELL TO CHANGE WHERE RESULTS ARE SAVED**

In [1]:
import os
from pathlib import Path

# ============================================================================
# OUTPUT FOLDER CONFIGURATION - CHANGE THIS PATH
# ============================================================================

OUTPUT_FOLDER = r'C:\Box\Box\BOX Subhashree Singh\Business\PAS\Output'

# Create folder if it doesn't exist
Path(OUTPUT_FOLDER).mkdir(parents=True, exist_ok=True)

print(f'✓ Output folder: {OUTPUT_FOLDER}')
print(f'✓ Folder exists: {os.path.exists(OUTPUT_FOLDER)}')
print()
print('Results will be saved to:')
print(f'  • feature_importance_analysis_notebook.xlsx')
print(f'  • feature_importance_analysis.json')

✓ Output folder: C:\Box\Box\BOX Subhashree Singh\Business\PAS\Output
✓ Folder exists: True

Results will be saved to:
  • feature_importance_analysis_notebook.xlsx
  • feature_importance_analysis.json


## Setup: Install Dependencies

In [2]:
import subprocess
import sys

try:
    import pandas as pd
    import numpy as np
    print('✓ pandas and numpy already installed')
except ImportError:
    print('Installing pandas and numpy...')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'pandas', 'numpy', 'openpyxl'])
    print('✓ Installation complete')

✓ pandas and numpy already installed


## Import Libraries

In [3]:
import pandas as pd
import numpy as np
from typing import Dict

print('✓ Libraries imported successfully')

✓ Libraries imported successfully


## Configuration: PAS Model Weights

**⚙️ MODIFY THESE VALUES TO TEST DIFFERENT SCENARIOS**

In [4]:
# ============================================================================
# COMPOSITE-LEVEL WEIGHTS (change these to test scenarios)
# ============================================================================

COMPOSITE_WEIGHTS = {
    'adequacy': 0.40,        # ← Try: 0.50 to increase importance
    'capacity': 0.25,
    'appetite': 0.25,
    'environment': 0.10      # ← Try: 0.00 to drop entirely
}

print('Composite Weights:')
for comp, weight in COMPOSITE_WEIGHTS.items():
    print(f'  {comp:15s}: {weight:.0%}')
print(f'  Total: {sum(COMPOSITE_WEIGHTS.values()):.0%}')

Composite Weights:
  adequacy       : 40%
  capacity       : 25%
  appetite       : 25%
  environment    : 10%
  Total: 100%


In [5]:
# ============================================================================
# COMPONENT-LEVEL VARIABLE WEIGHTS
# ============================================================================

ADEQUACY_VARS = {
    'total_loss_cost': 0.20,
    'indemnity_loss_cost': 0.10,
    'expense_loss_cost': 0.05,
    'total_frequency': 0.10,
    'indemnity_frequency': 0.05,
    'total_severity': 0.05,
    'actual_loss_ratio': 0.10,
    'loss_free_years': 0.10,
    'limits_to_premium': 0.10,
}

CAPACITY_VARS = {
    'per_occurrence_limit': 0.20,
    'aggregate_limit': 0.10,
    'risk_count': 0.10,
}

APPETITE_VARS = {
    'specialty_risk_tier': 0.25,
    'state_venue_risk': 0.15,
    'years_since_graduation': 0.10,
    'rvu_ratio': 0.10,
    'tenure_with_carrier': 0.05,
    'hospital_rating': 0.05,
    'practice_size': 0.05,
}

ENVIRONMENT_VARS = {
    'income_inequality': 0.05,
    'population_density': 0.05,
    'violent_crime_rate': 0.03,
    'pct_uninsured': 0.02,
}

print(f'Adequacy variables: {len(ADEQUACY_VARS)}')
print(f'Capacity variables: {len(CAPACITY_VARS)}')
print(f'Appetite variables: {len(APPETITE_VARS)}')
print(f'Environment variables: {len(ENVIRONMENT_VARS)}')
print(f'Total: {len(ADEQUACY_VARS) + len(CAPACITY_VARS) + len(APPETITE_VARS) + len(ENVIRONMENT_VARS)} variables')

Adequacy variables: 9
Capacity variables: 3
Appetite variables: 7
Environment variables: 4
Total: 23 variables


## Helper Functions

In [6]:
def weighted_avg(values: Dict[str, float], weights: Dict[str, float]) -> float:
    """
    Compute weighted average with weight-zeroing for missing values.
    Missing values (NaN) zero out their weight and redistribute.
    """
    total_weighted = 0.0
    total_active_weight = 0.0
    
    for key, value in values.items():
        if key in weights and pd.notna(value):
            w = weights[key]
            total_weighted += value * w
            total_active_weight += w
    
    if total_active_weight == 0:
        return np.nan
    
    return total_weighted / total_active_weight


def compute_composite(adequacy: float, capacity: float, appetite: float, 
                     environment: float, weights: Dict[str, float]) -> float:
    """
    Compute composite score from component scores.
    """
    components = {
        'adequacy': adequacy,
        'capacity': capacity,
        'appetite': appetite,
        'environment': environment
    }
    return weighted_avg(components, weights)

print('✓ Helper functions defined')

✓ Helper functions defined


## 1. BASELINE SCENARIO

All components at 5.0 (portfolio average)

In [7]:
baseline_components = {
    'adequacy': 5.0,
    'capacity': 5.0,
    'appetite': 5.0,
    'environment': 5.0,
}

baseline_composite = compute_composite(
    baseline_components['adequacy'],
    baseline_components['capacity'],
    baseline_components['appetite'],
    baseline_components['environment'],
    COMPOSITE_WEIGHTS
)

print(f'Baseline Composite Score: {baseline_composite:.2f}')
print(f'\nComponent Scores:')
for comp, score in baseline_components.items():
    print(f'  {comp:15s}: {score:.1f}')

Baseline Composite Score: 5.00

Component Scores:
  adequacy       : 5.0
  capacity       : 5.0
  appetite       : 5.0
  environment    : 5.0


## 2. COMPONENT SENSITIVITY (Elasticity)

How much does the composite score change per 1% change in each component?

In [8]:
elasticity = {}

for comp_name in COMPOSITE_WEIGHTS.keys():
    scenario = baseline_components.copy()
    baseline_value = scenario[comp_name]
    scenario[comp_name] = baseline_value * 1.01
    
    new_composite = compute_composite(
        scenario['adequacy'],
        scenario['capacity'],
        scenario['appetite'],
        scenario['environment'],
        COMPOSITE_WEIGHTS
    )
    
    delta_pct = ((new_composite - baseline_composite) / baseline_composite) * 100
    elasticity[comp_name] = delta_pct

elasticity_df = pd.DataFrame(list(elasticity.items()), 
                             columns=['Component', 'Elasticity (%/%)'])
elasticity_df = elasticity_df.sort_values('Elasticity (%/%)', ascending=False)

print('\nComponent Elasticity:')
print(elasticity_df.to_string(index=False))
print(f'\n→ {elasticity_df.iloc[0]["Component"].title()} is most sensitive')


Component Elasticity:
  Component  Elasticity (%/%)
   adequacy              0.40
   capacity              0.25
   appetite              0.25
environment              0.10

→ Adequacy is most sensitive


## 3. COMPONENT IMPACT MATRIX

Absolute change in composite score for various % changes in each component

In [9]:
changes = [-50, -25, -10, 10, 25, 50]
components = list(COMPOSITE_WEIGHTS.keys())

matrix = {}
for pct in changes:
    col_data = {}
    for comp in components:
        scenario = baseline_components.copy()
        baseline_value = scenario[comp]
        scenario[comp] = baseline_value * (1 + pct / 100)
        
        new_composite = compute_composite(
            scenario['adequacy'],
            scenario['capacity'],
            scenario['appetite'],
            scenario['environment'],
            COMPOSITE_WEIGHTS
        )
        
        delta = new_composite - baseline_composite
        col_data[comp] = delta
    
    matrix[f'{pct:+d}%'] = col_data

impact_df = pd.DataFrame(matrix).T
impact_df = impact_df[components]

print('\nComponent Impact Matrix (Absolute Score Change):')
print(impact_df.round(3).to_string())


Component Impact Matrix (Absolute Score Change):
      adequacy  capacity  appetite  environment
-50%      -1.0    -0.625    -0.625       -0.250
-25%      -0.5    -0.312    -0.312       -0.125
-10%      -0.2    -0.125    -0.125       -0.050
+10%       0.2     0.125     0.125        0.050
+25%       0.5     0.312     0.312        0.125
+50%       1.0     0.625     0.625        0.250


## 4. VARIABLE STRUCTURAL IMPORTANCE

Rank variables by their direct impact on the composite score.

In [10]:
variable_importance = {}

# Adequacy
for var_name, var_weight in ADEQUACY_VARS.items():
    direct_effect = var_weight * COMPOSITE_WEIGHTS['adequacy']
    variable_importance[var_name] = {
        'component': 'adequacy',
        'variable_weight': var_weight,
        'component_weight': COMPOSITE_WEIGHTS['adequacy'],
        'direct_effect': direct_effect,
    }

# Capacity
for var_name, var_weight in CAPACITY_VARS.items():
    direct_effect = var_weight * COMPOSITE_WEIGHTS['capacity']
    variable_importance[var_name] = {
        'component': 'capacity',
        'variable_weight': var_weight,
        'component_weight': COMPOSITE_WEIGHTS['capacity'],
        'direct_effect': direct_effect,
    }

# Appetite
for var_name, var_weight in APPETITE_VARS.items():
    direct_effect = var_weight * COMPOSITE_WEIGHTS['appetite']
    variable_importance[var_name] = {
        'component': 'appetite',
        'variable_weight': var_weight,
        'component_weight': COMPOSITE_WEIGHTS['appetite'],
        'direct_effect': direct_effect,
    }

# Environment
for var_name, var_weight in ENVIRONMENT_VARS.items():
    direct_effect = var_weight * COMPOSITE_WEIGHTS['environment']
    variable_importance[var_name] = {
        'component': 'environment',
        'variable_weight': var_weight,
        'component_weight': COMPOSITE_WEIGHTS['environment'],
        'direct_effect': direct_effect,
    }

var_df = pd.DataFrame([
    {
        'Variable': k,
        'Component': v['component'],
        'Variable Weight': v['variable_weight'],
        'Component Weight': v['component_weight'],
        'Direct Effect': v['direct_effect'],
    }
    for k, v in variable_importance.items()
])

var_df['Rank'] = var_df['Direct Effect'].rank(ascending=False).astype(int)
var_df = var_df.sort_values('Rank')[['Rank', 'Variable', 'Component', 'Variable Weight', 'Component Weight', 'Direct Effect']]

print('\nVariable Importance Ranking (Top 15):')
print(var_df.head(15).to_string(index=False))
print(f'\n... and {len(var_df) - 15} more variables')


Variable Importance Ranking (Top 15):
 Rank               Variable Component  Variable Weight  Component Weight  Direct Effect
    1        total_loss_cost  adequacy             0.20              0.40         0.0800
    2    specialty_risk_tier  appetite             0.25              0.25         0.0625
    3   per_occurrence_limit  capacity             0.20              0.25         0.0500
    6    indemnity_loss_cost  adequacy             0.10              0.40         0.0400
    6      actual_loss_ratio  adequacy             0.10              0.40         0.0400
    6      limits_to_premium  adequacy             0.10              0.40         0.0400
    6        total_frequency  adequacy             0.10              0.40         0.0400
    6        loss_free_years  adequacy             0.10              0.40         0.0400
    9       state_venue_risk  appetite             0.15              0.25         0.0375
   11             risk_count  capacity             0.10              0.

## 5. DEAD WEIGHT DETECTION

Identify variables with minimal importance (candidates for removal)

In [11]:
total_importance = var_df['Direct Effect'].sum()
threshold_pct = 1.0
threshold = (threshold_pct / 100) * total_importance

dead_weight = var_df[var_df['Direct Effect'] < threshold].copy()
dead_weight['Importance %'] = (dead_weight['Direct Effect'] / total_importance * 100).round(2)
dead_weight = dead_weight.sort_values('Direct Effect', ascending=False)

print(f'\nDead Weight Detection (< {threshold_pct}% importance):')
print()

if len(dead_weight) > 0:
    print(dead_weight[['Variable', 'Component', 'Direct Effect', 'Importance %']].to_string(index=False))
    print(f'\nTotal: {len(dead_weight)} variables')
    print(f'Combined importance: {dead_weight["Importance %"].sum():.2f}%')
    print('\n→ These are candidates for removal')
else:
    print('No dead weight variables found')


Dead Weight Detection (< 1.0% importance):

          Variable   Component  Direct Effect  Importance %
 income_inequality environment          0.005          0.78
population_density environment          0.005          0.78
violent_crime_rate environment          0.003          0.47
     pct_uninsured environment          0.002          0.31

Total: 4 variables
Combined importance: 2.34%

→ These are candidates for removal


## 6. COMPONENT EFFICIENCY ANALYSIS

In [12]:
component_analysis = {}

all_vars = {
    'adequacy': ADEQUACY_VARS,
    'capacity': CAPACITY_VARS,
    'appetite': APPETITE_VARS,
    'environment': ENVIRONMENT_VARS,
}

for comp, var_dict in all_vars.items():
    var_weights = list(var_dict.values())
    component_analysis[comp] = {
        'Weight (%)': COMPOSITE_WEIGHTS[comp] * 100,
        '# Variables': len(var_dict),
        'Avg Var Wt': np.mean(var_weights),
        'Max Var Wt': max(var_weights),
        'Min Var Wt': min(var_weights),
    }

comp_df = pd.DataFrame(component_analysis).T
comp_df['Weight (%)'] = comp_df['Weight (%)'].apply(lambda x: f'{x:.1f}%')
comp_df = comp_df.reset_index().rename(columns={'index': 'Component'})

print('\nComponent Efficiency Analysis:')
print(comp_df.to_string(index=False))


Component Efficiency Analysis:
  Component Weight (%)  # Variables  Avg Var Wt  Max Var Wt  Min Var Wt
   adequacy      40.0%          9.0    0.094444        0.20        0.05
   capacity      25.0%          3.0    0.133333        0.20        0.10
   appetite      25.0%          7.0    0.107143        0.25        0.05
environment      10.0%          4.0    0.037500        0.05        0.02


## 7. EXPORT RESULTS TO EXCEL & JSON

**Results will be saved to your configured output folder**

In [13]:
import json

# ============================================================================
# EXPORT TO EXCEL
# ============================================================================

excel_file = os.path.join(OUTPUT_FOLDER, 'feature_importance_analysis_notebook.xlsx')

with pd.ExcelWriter(excel_file, engine='openpyxl') as writer:
    # Summary
    summary_data = {
        'Metric': [
            'Baseline Composite (all=5.0)',
            'Elasticity - Adequacy',
            'Elasticity - Capacity',
            'Elasticity - Appetite',
            'Elasticity - Environment',
        ],
        'Value': [
            f'{baseline_composite:.2f}',
            f'{elasticity["adequacy"]:.4f}',
            f'{elasticity["capacity"]:.4f}',
            f'{elasticity["appetite"]:.4f}',
            f'{elasticity["environment"]:.4f}',
        ]
    }
    pd.DataFrame(summary_data).to_excel(writer, sheet_name='Summary', index=False)
    
    # Impact Matrix
    impact_df.to_excel(writer, sheet_name='Impact Matrix')
    
    # Variable Importance
    var_df.to_excel(writer, sheet_name='Variable Importance', index=False)
    
    # Dead Weight
    if len(dead_weight) > 0:
        dead_weight[['Rank', 'Variable', 'Component', 'Direct Effect', 'Importance %']].to_excel(
            writer, sheet_name='Dead Weight', index=False
        )
    
    # Component Analysis
    comp_df.to_excel(writer, sheet_name='Component Analysis', index=False)

print(f'✓ Excel file saved:')
print(f'  {excel_file}')

# ============================================================================
# EXPORT TO JSON
# ============================================================================

json_file = os.path.join(OUTPUT_FOLDER, 'feature_importance_analysis.json')

results_json = {
    'baseline_composite': float(baseline_composite),
    'component_elasticity': {k: float(v) for k, v in elasticity.items()},
    'component_impact_matrix': impact_df.to_dict(),
    'variable_importance': {
        k: {**v, 'direct_effect': float(v['direct_effect']), 'variable_weight': float(v['variable_weight']), 'component_weight': float(v['component_weight'])}
        for k, v in variable_importance.items()
    },
}

with open(json_file, 'w') as f:
    json.dump(results_json, f, indent=2)

print(f'\n✓ JSON file saved:')
print(f'  {json_file}')

print(f'\n✓ All results saved to: {OUTPUT_FOLDER}')

✓ Excel file saved:
  C:\Box\Box\BOX Subhashree Singh\Business\PAS\Output\feature_importance_analysis_notebook.xlsx

✓ JSON file saved:
  C:\Box\Box\BOX Subhashree Singh\Business\PAS\Output\feature_importance_analysis.json

✓ All results saved to: C:\Box\Box\BOX Subhashree Singh\Business\PAS\Output


## KEY FINDINGS

### Summary
- **Adequacy dominates** (40% weight)
- **Capacity & Appetite equally important** (25% each)
- **Environment is underutilized** (10% with weak variables)
- **Top variable**: total_loss_cost (8% direct impact)
- **Dead weight**: All 4 environment variables together < 2%

### Recommendations
1. Focus tuning on Adequacy (controls 40% of score)
2. Consider dropping Environment entirely if variables remain weak
3. Verify specialty_risk_tier and state_venue_risk (high impact)
4. Add stronger environment variables when available


## Quick Tips for Next Run

### To test different weights:
1. Go to **Configuration** section (cell with COMPOSITE_WEIGHTS)
2. Change the values
3. Re-run all cells below it (Shift+Enter)
4. New results will be saved automatically

### Example changes to try:
- Drop Environment: `'environment': 0.00`
- Increase Adequacy: `'adequacy': 0.50`
- Add new variable to ENVIRONMENT_VARS

### Files created:
- `feature_importance_analysis_notebook.xlsx` (Excel with 5 sheets)
- `feature_importance_analysis.json` (Raw data for programmatic use)
